# 03 · PROCESS — Limpieza documentada con log de auditoría
Cada paso registra conteos antes/después en `outputs/tables/log_limpieza_datos.txt`.

In [1]:
import sys, warnings, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
from IPython.display import Image, display
from src.data_loader import construir_dataset_maestro
df = construir_dataset_maestro()

  ✓ === INICIO PROCESO — Base atunera Manabí (SUPERCIAS) + Bangkok + ONI ===
  ✓ [1] Panel SUPERCIAS cargado: 2,983 empresa-año, 336 empresas, 2010–2025
  ✓ [2] Costo de ventas extraído (códigos {'NIIF_jerarquico': ['501', '51'], 'casilleros': ['797', '7991']}): disponible en 56.2% de registros
  ✓ [3] Tipo empresa: {'Pesquera': 2291, 'Manufacturera': 692} (Manabí: Manta/Montecristi/Jaramijó; CIIU A0311 → Pesquera, C1020 → Manufacturera)
  ✓ [4] Duplicados ruc-año eliminados (se conserva el registro más completo) | 2,983 → 2,983 (0 eliminados, 0.0%)
  ✓ [5] Excluidas empresa-año sin activos o sin ingresos (inactivas / 'de papel') | 2,983 → 2,086 (897 eliminados, 30.1%)
  ✓ [6] Ratios calculados. Margen bruto disponible: 1,648 obs
  ✓ [7] Outliers |z|>3 en margen_bruto: 2 (conservados y marcados); winsorización p1–p99 en margen_bruto_w [-1.493, 0.918]
  ✓ [7] Outliers |z|>3 en margen_operativo: 1 (conservados y marcados); winsorización p1–p99 en margen_operativo_w [-35.964, 1.000]
  ✓ [

  ✓ [10] Precio Bangkok anual unido (2011–2026). Obs sin precio (2010): 70
  ✓ [11] ONI anual unido (promedio, máx., mín.)
  ✓ [12] Dataset maestro: 2,086 obs empresa-año, 288 empresas, 16 años (2010–2025)


In [2]:
print(open('../outputs/tables/log_limpieza_datos.txt', encoding='utf-8').read())

2026-10-07 16:57:07,105 | PASO: === INICIO PROCESO — Base atunera Manabí (SUPERCIAS) + Bangkok + ONI ===
2026-10-07 16:57:07,132 | PASO: [1] Panel SUPERCIAS cargado: 2,983 empresa-año, 336 empresas, 2010–2025
2026-10-07 16:57:07,255 | PASO: [2] Costo de ventas extraído (códigos {'NIIF_jerarquico': ['501', '51'], 'casilleros': ['797', '7991']}): disponible en 56.2% de registros
2026-10-07 16:57:07,258 | PASO: [3] Tipo empresa: {'Pesquera': 2291, 'Manufacturera': 692} (Manabí: Manta/Montecristi/Jaramijó; CIIU A0311 → Pesquera, C1020 → Manufacturera)
2026-10-07 16:57:07,261 | PASO: [4] Duplicados ruc-año eliminados (se conserva el registro más completo) | 2,983 → 2,983 (0 eliminados, 0.0%)
2026-10-07 16:57:07,265 | PASO: [5] Excluidas empresa-año sin activos o sin ingresos (inactivas / 'de papel') | 2,983 → 2,086 (897 eliminados, 30.1%)
2026-10-07 16:57:07,274 | PASO: [6] Ratios calculados. Margen bruto disponible: 1,648 obs
2026-10-07 16:57:07,277 | PASO: [7] Outliers |z|>3 en margen_bru

In [3]:
display(df.groupby(['anio','tipo_empresa']).ruc.nunique().unstack())
display(df[['margen_bruto','margen_operativo','margen_neto','roa','roe']].describe(percentiles=[.05,.5,.95]).round(3))

tipo_empresa,Manufacturera,Pesquera
anio,,
2010,26,44
2011,23,54
2012,28,65
2013,31,78
2014,32,83
2015,29,90
2016,29,89
2017,32,110
2018,31,112


,margen_bruto,margen_operativo,margen_neto,roa,roe
count,1648.000,1981.000,2086.000,2086.000,1973.000
mean,-102.442,-2802.431,-2589.427,-0.920,-0.069
std,3722.816,105826.787,103078.985,30.409,9.848
min,-149999.000,-4693854.667,-4693854.667,-1303.305,-291.663
5%,-0.001,-0.407,-0.361,-0.139,-0.318
50%,0.156,0.019,0.011,0.012,0.055
95%,0.654,0.315,0.215,0.298,0.889
max,0.999,4.202,18.296,1.486,250.315


### Decisiones de limpieza (y por qué)
1. **No se imputan ratios.** Interpolar márgenes inventaría rentabilidad que no existió.
2. **Outliers se marcan, no se eliminan**; para estimar se acotan los ratios a [−1, 1] y se exige ingresos ≥ USD 100 mil.
3. **Sin efectos fijos de año en las regresiones:** el precio anual es igual para todas las empresas → colinealidad perfecta.
4. IPC Ecuador del protocolo usado solo para niveles en USD constantes (**pendiente validar con INEC**); los ratios no requieren deflactación.